# ЛР №3 — Теория к лабораторной работе
## «Синхронизация потоков и времени в распределённом приложении»

**Предметная область (вариант 14):** «Заказ и доставка цветов» — продолжение ЛР №1 и ЛР №2.
**Общий ресурс:** `Bouquet.AvailableCount` — количество доступных экземпляров букета.
**Проект:** `Лабы\Лаб3` (Server, Server.Semaphore, Client, Shared, Server.Tests).

> **ЛР №2 и ЛР №3 сдаются отдельно друг от друга.** В ЛР №2 Race Condition был
> намеренно **не** исправлен — требовалось показать его последствия. В ЛР №3 эта
> проблема устраняется механизмами синхронизации, а вторая часть работы посвящена
> синхронизации времени клиента с сервером с учётом задержки передачи данных.

### Содержание

| № | Раздел | Что доказывает |
|---|---|---|
| 1 | Race Condition: read → check → update | почему конкурентный доступ ломает счётчик ресурса |
| 2 | Критическая секция и взаимное исключение | какие действия должны быть неделимыми |
| 3 | Механизмы синхронизации в C# | lock, Monitor, Mutex, SemaphoreSlim, Interlocked |
| 4 | Реализация через `lock` | вход/выход из критической секции, что именно защищено |
| 5 | Правило: проверка внутри секции | почему `check()` вне `lock` возвращает гонку |
| 6 | Реализация через `SemaphoreSlim` | альтернативный проект, сравнение с `lock` |
| 7 | Журнал LOCK/UNLOCK/READ/WRITE | доказательство взаимного исключения по меткам времени |
| 8 | Границы критической секции | почему нельзя блокировать весь метод |
| 9 | Тесты синхронизации | 1 SUCCESS + 1 FAIL при одном ресурсе |
| 10 | Проблема часов в распределённой системе | skew и drift, модель рассинхронизации |
| 11 | TIME_REQUEST / TIME_RESPONSE | получение времени сервера |
| 12 | RTT, Offset, CorrectedTime | формулы и реализация поправки |
| 13 | Влияние задержки и сравнение с NTP | почему точность падает на `delay/2` |
| 14 | Порядок экспериментов и таблицы | протокол для защиты и заполнения отчёта |
| 15 | Ответы на контрольные вопросы | краткие формулировки для защиты

---
## 1. Race Condition: read → check → update (что было в ЛР №2)

**Race Condition** — ситуация, при которой результат работы программы зависит от порядка
и момента выполнения нескольких параллельных операций над общими данными.

В ЛР №2 сервер обрабатывал каждого клиента в отдельной `Task`, и общий ресурс
(`Bouquet.AvailableCount`) был доступен всем. Операция бронирования состояла из трёх шагов:

```
Шаг 1: READ   available = AvailableCount     ← чтение общего ресурса
Шаг 2: DELAY  окно гонки (искусственная задержка)
Шаг 3: WRITE  AvailableCount--               ← изменение ресурса
```

Между шагом 1 и шагом 3 ресурс может изменить другая задача — именно в этом и состоит гонка.

In [ ]:
// Server/BusinessLogic.cs — режим SyncMechanism.None (поведение ЛР №2, сохранено для сравнения)
private async Task<Response> ReserveWithoutSyncAsync(Bouquet bouquet, int bouquetId, int criticalDelayMs,
    RequestContext? context, CancellationToken ct)
{
    // Шаг 1: ПРОВЕРКА — чтение общего ресурса в локальную переменную (без блокировки)
    int available = bouquet.AvailableCount;
    Log(context, "READ", $"Букет #{bouquetId} — доступно {available} шт. (без синхронизации)");

    // Шаг 2: окно гонки — между проверкой и изменением проходит время
    if (criticalDelayMs > 0)
    {
        await Task.Delay(criticalDelayMs, ct);
    }

    // Шаг 3: РЕШЕНИЕ принимается по УСТАРЕВШЕМУ значению
    if (available <= 0)
    {
        Log(context, "FAIL", $"Букет #{bouquetId} — отказ: свободных экземпляров не осталось");
        throw new OperationException(ErrorCodes.OutOfStock, $"Букет «{bouquet.Name}» недоступен");
    }

    bouquet.AvailableCount--;   // запись без синхронизации — возможна потеря обновления
    Log(context, "WRITE", $"Букет #{bouquetId} — установлено {bouquet.AvailableCount} шт.");
    return Response.Ok(new { bouquetId, bouquet.Name, AvailableAfter = bouquet.AvailableCount });
}

**Что происходит при `AvailableCount = 1` и двух одновременных заявках** (окно гонки 2 с):

```
Client 1                         Client 2
────────                         ────────
READ  available = 1             (ждёт)
      ↓ окно гонки
                                 READ  available = 1        ← оба увидели «1»
      ↓                                  ↓
WRITE available: 1 → 0          WRITE available: 1 → −1    ← оба считали, что место свободно
SUCCESS                          SUCCESS                     ← ресурс был один, а успехов два
```

Итог: **оба клиента получили SUCCESS при одном свободном букете**, а счётчик ушёл в минус
(`AvailableCount = −1`). Это и есть **потерянное обновление** (lost update): решение
принималось по значению, которое к моменту записи уже не было актуальным.

**Вывод, который требуется от ЛР №3:** защитить нужно не отдельную строку, а
**последовательность** «прочитать → проверить → изменить», и сделать её неделимой.

---
## 2. Критическая секция и взаимное исключение

**Критическая секция** — участок программы, в котором выполняются действия с общим
ресурсом, не допускающие одновременного выполнения несколькими потоками.

**Взаимное исключение (mutual exclusion)** — требование: в каждый момент времени критическую
секцию выполняет не более одного потока (задачи). Если оно выполнено, Race Condition
для этого участка исключена.

В нашей задаче критическая секция — это атомарная бизнес-операция:

```
Проверить наличие ресурса
        ↓
Изменить состояние ресурса
        ↓
Подтвердить операцию
```

Требования к границам секции:

* **минимум** — всё, что читает или пишет общее состояние (`AvailableCount`);
* **максимум** — только эти действия; блокировать весь метод нельзя (см. раздел 8);
* **всё, что должно быть атомарным, лежит в одной секции** (Требование 5): и проверка,
  и изменение — иначе операция снова станет неатомарной.

---
## 3. Механизмы синхронизации в C#

| Механизм | Что даёт | Особенности |
|---|---|---|
| `lock` | взаимное исключение в пределах процесса | компилятор разворачивает в `Monitor.Enter/Exit`; внутри нельзя `await` |
| `Monitor` | то же, но низкоуровнево: вход/выход вручную | нужен `try/finally`, поддерживает ожидание с таймаутом |
| `Mutex` | взаимное исключение **между процессами** |named-объект в ОС, заметно большие накладные расходы, чем у `lock` |
| `SemaphoreSlim` | счётчик разрешений (1 → взаимное исключение, N → ограничение параллелизма) | асинхронный (`WaitAsync`), внутри допустим `await` |
| `Interlocked` | атомарные операции над одной переменной (`Increment`, `Exchange`, `CompareExchange`) | не заменяет `lock`, если нужно атомарно выполнить несколько действий |

In [ ]:
// Server/BusinessLogic.cs — Требование 5: lock как основной механизм
private readonly object _syncObject = new();   // объект синхронизации (только для блокировки)

private Response ReserveWithLock(Bouquet bouquet, int bouquetId, int criticalDelayMs, RequestContext? context)
{
    lock (_syncObject)   // ← вход в критическую секцию (взаимное исключение)
    {
        Log(context, "LOCK", $"Букет #{bouquetId} — вход в критическую секцию (lock)");
        try
        {
            // ПРОВЕРКА и ИЗМЕНЕНИЕ в ОДНОЙ критической секции (Требование 6)
            Log(context, "READ", $"Букет #{bouquetId} — доступно {bouquet.AvailableCount} шт.");
            if (criticalDelayMs > 0)
            {
                Thread.Sleep(criticalDelayMs);   // lock — синхронный: поток удерживается
            }

            return CheckAndUpdate(bouquet, bouquetId, context);   // решение + запись
        }
        finally
        {
            Log(context, "UNLOCK", $"Букет #{bouquetId} — выход из критической секции (lock)");
        }
    }
}

**Ядро критической секции** — общий для всех режимов метод `CheckAndUpdate`: он
принимает решение по **актуальному** значению ресурса и сразу же изменяет его.
Вызывать его можно только изнутри секции — это и обеспечивает атомарность:

```csharp
private static Response CheckAndUpdate(Bouquet bouquet, int bouquetId, RequestContext? context)
{
    if (bouquet.AvailableCount <= 0)
    {
        Log(context, "FAIL", $"Букет #{bouquetId} — отказ: свободных экземпляров не осталось");
        throw new OperationException(ErrorCodes.OutOfStock, $"Букет «{bouquet.Name}» недоступен");
    }

    bouquet.AvailableCount--;
    Log(context, "WRITE", $"Букет #{bouquetId} — установлено {bouquet.AvailableCount} шт.");
    Log(context, "SUCCESS", $"Букет #{bouquetId} — бронирование подтверждено");
    return Response.Ok(new { bouquetId, bouquet.Name, AvailableAfter = bouquet.AvailableCount });
}
```

### 3.1 Что важно понимать про `lock`

* `lock` — это **синхронный** примитив: вошедший поток **удерживается** до выхода, второй
  поток ждёт. Поэтому внутри `lock` нельзя использовать `await` (компилятор не позволит),
  а для имитации длительной работы используется `Thread.Sleep`.
* Объект синхронизации — отдельный `private readonly object`, а не `this` и не `typeof(BusinessLogic)`:
  иначе блокировку мог бы захватить посторонний код.
* `finally` гарантирует выход из секции даже при исключении — иначе ресурс «залипнет».
* `lock` синхронизирует **потоки одного процесса**; для разных процессов нужен `Mutex`.

In [ ]:
// Требование 6: разница между правильной и неправильной границей критической секции

// ПРАВИЛЬНО: проверка и изменение — в одной критической секции
lock (_syncObject)
{
    if (bouquet.AvailableCount <= 0) return Fail();   // проверка внутри
    bouquet.AvailableCount--;                         // изменение внутри
}

// НЕПРАВИЛЬНО: проверка ВЫНЕ блокировки — Race Condition возвращается,
// потому что между этими двумя строками ресурс может изменить другой поток
if (bouquet.AvailableCount <= 0) return Fail();     // ← проверка снаружи
lock (_syncObject)
{
    bouquet.AvailableCount--;                         // ← изменение под блокировкой
}

**Правило, которое нужно уметь объяснить на защите:** синхронизировать нужно не
отдельную операцию, а **всю последовательность действий, которая должна быть неделимой**.
Если проверка и изменение не попали в одну критическую секцию, между ними снова
возникает окно, в котором ресурс может изменить другая задача, — и гонка возвращается.

В нашей операции неделимы ровно три шага: **проверить → изменить → подтвердить**.
Чтение `Bouquet` из хранилища, формирование JSON-ответа и логирование входа/выхода
в блокировке не нужны.

In [ ]:
// Server/BusinessLogic.cs — Требование 9: SemaphoreSlim с одним разрешением
private readonly SemaphoreSlim _semaphore = new(1, 1);   // 1 разрешение = взаимное исключение

private async Task<Response> ReserveWithSemaphoreAsync(Bouquet bouquet, int bouquetId, int criticalDelayMs,
    RequestContext? context, CancellationToken ct)
{
    await _semaphore.WaitAsync(ct);   // ← вход в критическую секцию (поток НЕ блокируется)
    try
    {
        Log(context, "LOCK", $"Букет #{bouquetId} — вход в критическую секцию (SemaphoreSlim)");
        Log(context, "READ", $"Букет #{bouquetId} — доступно {bouquet.AvailableCount} шт.");
        if (criticalDelayMs > 0)
        {
            await Task.Delay(criticalDelayMs, ct);   // внутри секции await разрешён
        }

        return CheckAndUpdate(bouquet, bouquetId, context);
    }
    finally
    {
        Log(context, "UNLOCK", $"Букет #{bouquetId} — выход из критической секции (SemaphoreSlim)");
        _semaphore.Release();          // ← выход из критической секции (в finally!)
    }
}

### 4.1 Ответы на вопросы Требования 9

| Вопрос | Ответ |
|---|---|
| Что он синхронизирует? | Счётчик разрешений: одновременно внутри критической секции может быть не более N задач. При N = 1 это взаимное исключение |
| Может ли использоваться между процессами? | Нет. `SemaphoreSlim` — внутрипроцессный примитив; между процессами нужен `System.Threading.Semaphore` или `Mutex` |
| Подходит ли он для нашего сценария? | Да: доступ к `AvailableCount` нужен только внутри одного серверного процесса, а операция короткая |
| Чем отличается от `lock`? | `WaitAsync` не блокирует поток (задача «паркуется» и освобождает поток), внутри допустим `await`, счётчик разрешений может быть больше 1; `lock` — только взаимное исключение, внутри `await` запрещён |

In [ ]:
// Server/RequestContext.cs — контекст запроса: кто выполняет и куда писать журнал
public sealed class RequestContext
{
    public int ClientId { get; init; }                    // номер клиента (Client 1, Client 2, …)
    public Action<string, string>? Journal { get; init; }  // (фаза, описание): LOCK/UNLOCK/READ/WRITE/SUCCESS/FAIL
    public string ClientTag => $"Client {ClientId}";
}

// Server/Server.cs — контекст создаётся на каждый запрос и передаётся в обработчик
var context = new RequestContext
{
    ClientId = clientId,
    Journal = (phase, detail) => Log(clientId, phase, operation, detail: detail)
};

Response response = await _handler.HandleAsync(json, context);

// Server/Server.cs — вывод в консоль и в Journal (используется в unit-тестах)
private void Log(int clientId, string phase, string operation, double? elapsedMs = null, string? detail = null)
{
    var entry = new ProcessingLogEntry(DateTime.Now, clientId, phase, operation,
        Task.CurrentId, Environment.CurrentManagedThreadId, elapsedMs, detail);
    Journal.Enqueue(entry);
    Console.WriteLine($"[{entry.Timestamp:HH:mm:ss.fff}] Client {clientId} → {phase,-10} {operation,-22} …{tail}");
}

## 5. Журнал LOCK/UNLOCK/READ/WRITE — доказательство взаимного исключения

Реальный журнал эксперимента (букет #3, `AvailableCount = 1`, задержка 1500 мс, механизм `lock`):

```
[19:20:32.601] Client 2 → START      ReserveBouquet   Task=2 Thread=8
[19:20:32.602] Client 2 → LOCK       ReserveBouquet   Task=2 Thread=8  Букет #3 — вход в критическую секцию (lock)
[19:20:32.603] Client 2 → READ       ReserveBouquet   Task=2 Thread=8  Букет #3 — доступно 1 шт.
[19:20:32.825] Client 3 → START      ReserveBouquet   Task=3 Thread=9
[19:20:34.108] Client 2 → WRITE      ReserveBouquet   Task=2 Thread=8  Букет #3 — установлено 0 шт.
[19:20:34.114] Client 2 → SUCCESS    ReserveBouquet   Task=2 Thread=8  Букет #3 — бронирование подтверждено
[19:20:34.117] Client 2 → UNLOCK     ReserveBouquet   Task=2 Thread=8  Букет #3 — выход из критической секции (lock)
[19:20:34.117] Client 3 → LOCK       ReserveBouquet   Task=3 Thread=9  Букет #3 — вход в критическую секции (lock)
[19:20:34.118] Client 3 → READ       ReserveBouquet   Task=3 Thread=9  Букет #3 — доступно 0 шт.
[19:20:35.621] Client 3 → FAIL       ReserveBouquet   Task=3 Thread=9  Букет #3 — отказ: свободных экземпляров не осталось
[19:20:35.622] Client 3 → UNLOCK     ReserveBouquet   Task=3 Thread=9  Букет #3 — выход из критической секции (lock)
```

**Как это читается (два независимых вывода):**

1. **Параллельность сохранилась.** `START(Client 3)` в 32.825 раньше `END/UNLOCK(Client 2)`
   в 34.117 — второй клиент действительно работал одновременно с первым (задачи в
   разных потоках: `Task=2/Thread=8` и `Task=3/Thread=9`).
2. **Взаимное исключение обеспечено.** `LOCK(Client 3)` в 34.117 идёт **строго после**
   `UNLOCK(Client 2)` в 34.117. Пока первый был в секции, второй в неё не попал, поэтому
   прочитал уже `0` и получил FAIL. Интервалы `[LOCK, UNLOCK]` двух клиентов не пересекаются —
   именно это и проверяет тест `TwoClients_CriticalSections_DoNotOverlap`.

---
## 6. Границы критической секции: почему нельзя блокировать весь метод

На защите часто спрашивают: «а почему нельзя просто взять и обернуть весь метод в `lock`?»

**Причины:**

1. **Излишняя сериализация.** Внутри метода есть работа, не связанная с общим ресурсом
   (поиск букета в хранилище, формирование ответа). Если заблокировать её, запросы
   разных клиентов станут выполняться по очереди — сервер перестанет быть параллельным,
   и мы вернёмся к архитектуре ЛР №1.
2. **Риск взаимной блокировки (deadlock).** Если внутри секции вызывается другой метод,
   который тоже берёт ту же блокировку (или ждёт внешнего ресурса), возможен «тупик».
3. **Лишние задержки.** Чем длиннее секция, тем дольше остальные клиенты ждут.

**Правило:** синхронизируем минимально достаточный участок, в котором выполняется
несколько связанных действий над общим состоянием. Проверяется тестом
`SynchronizedReservations_DoNotBlockUnrelatedRequests`: пока два клиента соревнуются
за букет (секция занята ~1.5 с каждый), независимый запрос `CreateOrder` от третьего
клиента завершается менее чем за 1 с — значит, заблокирован только нужный участок.

Замечание об асинхронности: в режиме `lock` имитация длительной работы выполнена через
`Thread.Sleep`, потому что `await` внутри `lock` невозможен. В режиме `SemaphoreSlim`
тот же участок использует `await Task.Delay(...)` — и поток при этом не удерживается.

In [ ]:
// Server.Tests/SynchronizationTests.cs — доказательство исправления (Lock и SemaphoreSlim)
[Theory]
[InlineData(SyncMechanism.Lock)]
[InlineData(SyncMechanism.SemaphoreSlim)]
public async Task TwoClients_ReserveLastBouquet_OnlyOneSucceeds(SyncMechanism sync)
{
    using var host = new TestServerHost(sync);      // сервер на эфемерном порту
    using var admin = host.CreateClient();
    using var client1 = host.CreateClient();
    using var client2 = host.CreateClient();

    Response set = await admin.SendAsync(RequestBuilder.SetBouquetAvailability(1, 1));  // ресурс = 1
    Assert.True(set.Success, set.Message);

    Task<Response> task1 = client1.SendAsync(RequestBuilder.ReserveBouquet(1, criticalDelayMs: 1500));
    await Task.Delay(200);   // второй клиент входит в секцию, пока первый внутри
    Task<Response> task2 = client2.SendAsync(RequestBuilder.ReserveBouquet(1, criticalDelayMs: 1500));

    Response[] results = await Task.WhenAll(task1, task2);

    // ровно один успех, второй — отказ по исчерпанию ресурса
    Assert.Equal(1, results.Count(r => r.Success));
    Response failed = results.Single(r => !r.Success);
    Assert.Equal(ErrorCodes.OutOfStock, failed.ErrorCode);

    // ресурс не ушёл в минус: 1 − 1 = 0 (в ЛР №2 было −1)
    Response status = await admin.SendAsync(RequestBuilder.GetBouquetStatus(1));
    Assert.Equal(0, status.Data!.Value.GetProperty("availableCount").GetInt32());
}

**Конструкция теста как доказательство:**

* `Assert.Equal(1, results.Count(r => r.Success))` — при одном ресурсе и двух заявках
  успех ровно один; в ЛР №2 здесь было бы 2.
* `Assert.Equal(ErrorCodes.OutOfStock, failed.ErrorCode)` — отказ приходит именно по
  предметной причине (ресурс занят), а не из-за ошибки сервера.
* `Assert.Equal(0, availableCount)` — проверяется **состояние ресурса**: потерянного
  обновления больше нет.
* Тест parametrized (`[Theory]`): один и тот же сценарий проверяет оба механизма —
  `lock` и `SemaphoreSlim` должны вести себя одинаково.

Полный набор тестов части 1:

| Тест | Что доказывает |
|---|---|
| `TwoClients_ReserveLastBouquet_OnlyOneSucceeds` (Lock, SemaphoreSlim) | при 1 ресурсе ровно 1 SUCCESS, второй `OUT_OF_STOCK`, ресурс = 0 |
| `TwoClients_CriticalSections_DoNotOverlap` (Lock, SemaphoreSlim) | порядок `LOCK₁ → UNLOCK₁ → LOCK₂ → UNLOCK₂` — секции не пересекаются |
| `SynchronizedReservations_DoNotBlockUnrelatedRequests` | независимый запрос не ждёт блокировку |
| `…WithoutSync_BothSucceed_RaceConditionOccurs` | состояние «до»: оба SUCCESS, ресурс −1 |

### 6.1 Таблица 1 — сравнение результатов синхронизации общего ресурса

Шаблон (заполняется по меткам времени из журнала сервера):

| Механизм | Клиент | Время отправки запроса | Время входа в критическую секцию | Время выхода | Результат операции |
|---|---|---|---|---|---|
| Без синхронизации | Client 1 | | | | Success / Fail |
| Без синхронизации | Client 2 | | | | Success / Fail |
| lock | Client 1 | | | | Success / Fail |
| lock | Client 2 | | | | Success / Fail |
| SemaphoreSlim | Client 1 | | | | Success / Fail |
| SemaphoreSlim | Client 2 | | | | Success / Fail |

Заполненный пример (реальный запуск, задержка внутри секции 1500 мс):

| Механизм | Клиент | Отправка | LOCK | UNLOCK | Результат |
|---|---|---|---|---|---|
| Без синхронизации | Client 1 | 19:20:32.601 | — | — | **Success** |
| Без синхронизации | Client 2 | 19:20:32.825 | — | — | **Success** (ресурс стал −1 — гонка) |
| lock | Client 1 | 19:20:32.601 | 19:20:32.602 | 19:20:34.117 | **Success** |
| lock | Client 2 | 19:20:32.825 | 19:20:34.117 | 19:20:35.622 | **Fail (OUT_OF_STOCK)** |
| SemaphoreSlim | Client 1 | 19:21:18.35 | 19:21:18.361 | 19:21:19.871 | **Success** |
| SemaphoreSlim | Client 2 | 19:21:18.57 | 19:21:19.871 | 19:21:21.388 | **Fail (OUT_OF_STOCK)** |

**Вывод:** без синхронизации оба клиента успешны при одном ресурсе (Race Condition);
с `lock` и с `SemaphoreSlim` — ровно один успех, что и требуется доказать.

---
## 7. Часть 2. Проблема: часы разных узлов не совпадают

В распределённой системе каждый узел (сервер, каждый клиент) имеет собственные часы,
и они расходятся:

```
Server   → 10:00:00
Client 1 → 10:00:07     (часы спешат на +7 секунд)
Client 2 → 09:59:55     (часы отстают на −5 секунд)
```

Причины расхождения: разные источники синхронизации (осцилляторы разной точности),
сетевые задержки при автоматической настройке времени, дрейф (часы со временем
спешат/отстают всё сильнее). В терминах NTP это **skew** (смещение) и **drift** (уход).

**Последствие:** одинаковые события в разных узлах получают разные временные метки,
поэтому сравнить их и выстроить порядок событий по локальным часам невозможно.

Моделирование (Требование 1): **системное время компьютера не изменяется** — смещение
задаётся в программе параметром `--offset` и прибавляется к `UtcNow` при чтении часов.

In [ ]:
// Client/ClientClock.cs — модель часов клиента (Требование 1 и 4)
public sealed class ClientClock
{
    public TimeSpan ClockOffset { get; set; }        // искусственное смещение часов клиента
    public TimeSpan Correction { get; private set; }  // поправка, измеренная при синхронизации
    public bool IsSynchronized { get; private set; }

    // локальные часы: UtcNow + моделируемое смещение
    public DateTime LocalNow => DateTime.UtcNow + ClockOffset;

    // скорректированное время: CorrectedTime = LocalTime + Offset
    public DateTime CorrectedNow => LocalNow + Correction;

    public void ApplyCorrection(TimeSpan correction)
    {
        Correction = correction;
        IsSynchronized = true;
    }

    public void ResetCorrection() { Correction = TimeSpan.Zero; IsSynchronized = false; }
}

// Client/Program.cs — запуск клиента со смещёнными часами:
//   dotnet run --project Client -- --offset=+7      (часы спешат на 7 с)
//   dotnet run --project Client -- --offset=-5      (часы отстают на 5 с)

In [ ]:
// Server/BusinessLogic.cs — TIME_REQUEST / TIME_RESPONSE (Требование 3)
public Task<Response> GetServerTimeAsync(RequestContext? context = null)
{
    DateTime serverTime = DateTime.UtcNow;   // замер времени сервера
    Log(context, "TIME", $"Время сервера: {serverTime:HH:mm:ss.fff} UTC");

    return Task.FromResult(Response.Ok(new
    {
        serverTimeUtc = serverTime.ToString("o"),                                   // ISO-8601
        serverTimeUnixMs = new DateTimeOffset(serverTime).ToUnixTimeMilliseconds()  // Unix-мс
    }, $"Время сервера (UTC): {serverTime:HH:mm:ss.fff}"));
}

// Client/RequestBuilder.cs — запрос времени (delayMs — искусственная задержка сервера)
public static Request GetServerTime(int? delayMs = null)
{
    Dictionary<string, object?>? data = null;
    if (delayMs is not null) data = new Dictionary<string, object?> { ["delayMs"] = delayMs.Value };
    return Build("GetServerTime", data);
}

## 8. Обмен TIME_REQUEST / TIME_RESPONSE и демонстрация проблемы

Протокол синхронизации времени (упрощённый вариант):

```
Client                                     Server
──────                                     ──────
t1 = LocalNow                               
TIME_REQUEST (GetServerTime) ───────────►  
                                           serverTime = DateTime.UtcNow
◄────────────── TIME_RESPONSE {serverTime} 
t4 = LocalNow                               
```

Время передаётся двумя способами: `serverTimeUnixMs` (целое число, не зависит от
настроек локали — именно его читает клиент) и `serverTimeUtc` (ISO-8601, для читаемости).

**Демонстрация проблемы** (Требование 2) — пункт меню 9, клиент со смещением часов +7 с:

```
  Время сервера (UTC):          16:20:35.633
  Локальное время клиента:      16:20:42.632
  Смещение часов клиента:       +7.000 с (модель рассинхронизации)
  Разница Local − Server:       +7.000 с
```

Клиент не может определить «правильность» своих часов самостоятельно — единственный
источник истины в системе это часы сервера. Значит, нужно запросить время и вычислить
поправку, предварительно оценив задержку передачи данных.

In [ ]:
// Client/TimeSynchronizer.cs — синхронизация времени (Требования 3–6)
public sealed record TimeSyncResult(
    DateTime LocalBefore,            // t1 — локальное время перед отправкой запроса
    DateTime LocalAfter,             // t4 — локальное время после получения ответа
    DateTime ServerTime,             // serverTime из TIME_RESPONSE
    TimeSpan Rtt,                    // Round-Trip Time = t4 − t1
    TimeSpan EstimatedNetworkDelay,  // RTT / 2
    TimeSpan CalculatedOffset,       // Offset = serverTime − (t1 + RTT/2)
    DateTime CorrectedTime,          // LocalTime + Offset
    int ServerDelayMs);

public async Task<TimeSyncResult> SyncAsync(int? serverDelayMs = null, CancellationToken ct = default)
{
    DateTime t1 = _clock.LocalNow;                     // время отправки запроса
    Response response = await _client.SendAsync(RequestBuilder.GetServerTime(serverDelayMs), ct);
    DateTime t4 = _clock.LocalNow;                     // время получения ответа

    if (!response.Success)
        throw new TimeSynchronizationException($"Сервер не вернул время: {response.ErrorCode}");

    DateTime serverTime = ParseServerTime(response);

    TimeSpan rtt = t4 - t1;                            // Round-Trip Time
    TimeSpan networkDelay = rtt / 2;                   // RTT / 2 — задержка в одну сторону
    DateTime referenceLocal = t1 + networkDelay;       // момент сервера в шкале клиента
    TimeSpan offset = serverTime - referenceLocal;     // Offset = ServerTime − ClientTime

    _clock.ApplyCorrection(offset);                    // клиент сохраняет поправку
    return new TimeSyncResult(t1, t4, serverTime, rtt, networkDelay, offset,
        _clock.CorrectedNow, serverDelayMs ?? 0);
}

### 8.1 Формулы и их смысл (Требования 4–6)

```
RTT = t4 − t1                          # Round-Trip Time: полный обмен запрос-ответ
Network delay ≈ RTT / 2                # задержка в одну сторону (симметричное допущение)
момент сервера (в шкале клиента) = t1 + RTT / 2
Offset = serverTime − (t1 + RTT / 2)   # поправка, которую нужно прибавить к местным часам
CorrectedTime = LocalTime + Offset
```

**Почему именно так.** Ответ сервера приходит не мгновенно: пока сообщение шло по сети,
часы «шли дальше». Поэтому прямой поправки `serverTime − t1` было бы недостаточно — она
включала бы в себя и сетевую задержку. Если считать, что задержка в обе стороны примерно
одинакова, момент фиксации времени сервером приходится примерно на середину обмена:
`t1 + RTT/2`.

**Числовой пример** (смещение часов клиента +7 с, задержка сервера 0):

```
t1          = 16:20:42.632   (локальное, «спешащее» на 7 с)
serverTime  = 16:20:35.633   (время сервера)
t4          = 16:20:42.635   →  RTT = 0.003 с
t1 + RTT/2  = 16:20:42.634   (оценка момента сервера в шкале клиента)
Offset      = 16:20:35.633 − 16:20:42.634 = −7.001 с
Corrected   = 16:20:42.635 + (−7.001) = 16:20:35.634  ≈ время сервера
```

**Почему RTT не зависит от смещения часов клиента:** RTT — это *разность* двух показаний
одних и тех же часов, поэтому искусственное смещение (+7 с) входит в оба отсчёта и
в вычитании сокращается. Смещение влияет только на саму поправку `Offset`, но не на
оценку задержки.

## 9. Влияние задержки передачи данных (Требование 7)

Допущение «задержка в обе стороны одинакова» — ключевое. Если сервер задерживает
обработку запроса на `D` миллисекунд, отметка времени снимается позже, чем клиент
считает середину обмена:

```
t1 ──►[ сеть ]──► [ задержка D ] ──► serverTime ──►[ сеть ]──► t4
                 ↑
                 клиент считает, что serverTime ≈ t1 + RTT/2 (середина обмена)
```

**Ошибка поправки ≈ D/2** (если `D` заметно больше задержки сети).

Реальные измерения на этом компьютере (смещение часов клиента +7 с):

| Задержка сервера | RTT | RTT/2 | Вычисленная поправка | Остаточная ошибка |
|---|---|---|---|---|
| 0 мс | 0.003 с | 0.002 с | −7.001 с | **+0.001 с** |
| 1000 мс | 1.018 с | 0.509 с | −6.507 с | **−0.493 с** |

**Вывод:** при нулевой искусственной задержке поправка почти идеальна (ошибка 1 мс).
При задержке 1 с поправка ошибается примерно на +0.5 с (это и есть `delay/2`), и
скорректированное время «убегает» вперёд относительно серверного: остаточная ошибка
около −0.49 с. Чем больше задержка передачи, тем ниже точность синхронизации.

Практический вывод: NTP-серверы усредняют результаты нескольких обменов и отбрасывают
выбросы — в нашей лабораторной работе достаточно одного обмена, чтобы показать сам
принцип потери точности.

### 9.1 Сравнение с настоящим протоколом NTP

NTP измеряет **четыре** метки: `T1` (отправка), `T2` (приём сервером), `T3` (отправка
ответа), `T4` (получение клиентом) и вычисляет:

```
offset = ((T2 − T1) + (T3 − T4)) / 2      # смещение часов клиента
delay  = (T4 − T1) − (T3 − T2)            # суммарная задержка сети
```

Отличие от нашей упрощённой модели: используются **две** метки сервера, поэтому задержка
обработки `T3 − T2` исключается и не входит в ошибку. У нас сервер отдаёт одну метку,
поэтому задержка обработки попадает в результат — отсюда ошибка `≈ D/2`.

### 9.2 Почему нельзя добиться абсолютно одинаковых часов

1. **Асимметрия сети.** Задержка туда и обратно в общем случае различается, а мы
   измеряем только сумму (RTT).
2. **Джиттер (нестабильность задержки).** Значение RTT меняется от запроса к запросу,
   поэтому результат колеблется.
3. **Дрейф часов.** Поправка устаревает: часы продолжают идти с собственной скоростью,
   поэтому синхронизацию нужно повторять периодически.
4. **Разрешение и точность измерений.** Само измерение времени в .NET ограничено
   частотой таймера (обычно ~1–15 мс) и накладными расходами обработки.
5. **Обработка на сервере.** Время фиксируется в момент обработки, а не в момент
   прихода пакета — это добавляет неопределённость.

Поэтому синхронизация времени даёт **оценку с некоторой погрешностью**, а не абсолютную
гарантию: в наших измерениях остаточная ошибка после синхронизации без искусственных
задержек составляет порядка единиц миллисекунд.

### 9.3 Таблица 2 — результаты синхронизации времени

| Эксперимент | Локальное время клиента | Время сервера | Разница до синхронизации | RTT | Оценка задержки RTT/2 | Вычисленная поправка | Скорректированное время | Разница после синхронизации |
|---|---|---|---|---|---|---|---|---|
| 1 (без задержки) | 16:20:42.632 | 16:20:35.633 | +7.000 с | 0.003 с | 0.002 с | −7.001 с | 16:20:35.635 | +0.001 с |
| 2 (задержка 1000 мс) | 16:20:42.639 | 16:20:36.642 | +7.000 с | 1.018 с | 0.509 с | −6.507 с | 16:20:37.151 | −0.493 с |
| 3 (для проверки) | | | | | | | | |

**Вывод о точности.** Эксперимент 1 показывает, что без искусственных задержек алгоритм
почти точен (ошибка 1 мс). Эксперимент 2 показывает, что задержка обработки на сервере
снижает точность примерно до величины `delay/2` — это и есть цена сетевого обмена.

In [ ]:
// Server.Tests/TimeSynchronizationTests.cs — проверка синхронизации времени (Требование 11)
[Theory]
[InlineData(7)]    // часы клиента «спешат» на 7 с (пример из методички)
[InlineData(-5)]   // часы клиента «отстают» на 5 с
public async Task SyncAsync_WithClockSkew_ComputesOffsetAndCorrectsTime(double skewSeconds)
{
    using var host = new TestServerHost();
    using var client = host.CreateClient();

    TimeSpan skew = TimeSpan.FromSeconds(skewSeconds);
    var clock = new ClientClock { ClockOffset = skew };
    var synchronizer = new TimeSynchronizer(client, clock);

    TimeSyncResult result = await synchronizer.SyncAsync();

    // вычисленная поправка противоположна смещению часов: Offset ≈ −skew
    TimeSpan offsetError = result.CalculatedOffset + skew;
    Assert.True(Math.Abs(offsetError.TotalMilliseconds) < 500, "поправка неточна");

    // после синхронизации скорректированное время совпадает с временем сервера
    TimeSpan residual = await synchronizer.MeasureResidualErrorAsync();
    Assert.True(Math.Abs(residual.TotalMilliseconds) < 500, $"остаточная ошибка {residual}");

    // поправка сохранена в часах клиента (Требование 4)
    Assert.True(clock.IsSynchronized);
    Assert.Equal(result.CalculatedOffset, clock.Correction);
}

[Fact]
public async Task SyncAsync_WithServerDelay_ReducesAccuracy()
{
    // задержка 1500 мс увеличивает ошибку поправки примерно до delay/2
    TimeSyncResult result = await synchronizer.SyncAsync(serverDelayMs: 1500);
    TimeSpan offsetError = result.CalculatedOffset + skew;
    Assert.True(offsetError > TimeSpan.FromMilliseconds(300));   // ошибка заметна
    Assert.True(offsetError < TimeSpan.FromSeconds(1.2));        // но близка к delay/2
}

In [ ]:
# Сборка и тесты (33 теста: ЛР №1 + ЛР №2 + ЛР №3)
dotnet build FlowerShop.sln
dotnet test  FlowerShop.sln

# Терминал 1 — сервер (механизм синхронизации выбирается аргументом)
dotnet run --project Server -- 5000 --sync=lock         # основной вариант
dotnet run --project Server -- 5000 --sync=semaphore    # SemaphoreSlim в том же сервере
dotnet run --project Server -- 5000 --sync=none         # поведение ЛР №2 (Race Condition)

# Терминал 1 (альтернатива) — отдельный проект с SemaphoreSlim (Требование 9)
dotnet run --project Server.Semaphore -- 5000

# Терминал 2 и 3 — два клиента, моделирующие рассинхронизацию часов
dotnet run --project Client -- 127.0.0.1 5000 --offset=+7
dotnet run --project Client -- 127.0.0.1 5000 --offset=-5

## 10. Порядок проведения экспериментов (процедура для защиты)

### 10.1 Часть 1 — синхронизация общего ресурса

1. Запустить сервер (`--sync=lock`) и **два клиента**.
2. Client 1: пункт **7** (`SetAvailability`) — `bouquetId = 3`, `count = 1`.
3. Client 1: пункт **6** (`ReserveBouquet`), задержка 3000 мс (клиент внутри критической секции).
4. **Не дожидаясь ответа**, Client 2: пункт **6** с теми же параметрами.
5. В журнале сервера сравнить метки времени:
   * `START(Client 2)` < `UNLOCK(Client 1)` → обработка **параллельна**;
   * `LOCK(Client 2)` > `UNLOCK(Client 1)` → критические секции **не пересекаются**.
6. Результат операций: **Client 1 → SUCCESS, Client 2 → FAIL (OUT_OF_STOCK)**, ресурс = 0.

Для сравнения повторить с `--sync=none` (оба SUCCESS, ресурс −1) и с проектом
`Server.Semaphore` (тот же результат, что и `lock`).

### 10.2 Часть 2 — синхронизация времени

1. Запустить клиент с `--offset=+7` (часы спешат) и второй — с `--offset=-5`.
2. Пункт **9** — сравнить локальное время и время сервера, увидеть расхождение ±N с.
3. Пункт **10** — выполнить синхронизацию: показать Local time, Server time, RTT,
   RTT/2, Offset, Corrected time и остаточную ошибку.
4. Пункт **11** — повторить с искусственной задержкой сервера 1000 мс, сравнить точность.
5. Пункт **12** — проверить, что скорректированное время совпадает с новым замером
   времени сервера.
6. Пункт **14** — итоговый эксперимент: все проверки (общий ресурс + время + задержка)
   выполняются подряд, значения готовы для Таблицы 2.

## 11. Карта «требование → код → наблюдаемый результат»

| Требование | Реализация | Наблюдаемый результат |
|---|---|---|
| Тр. 2. Общий ресурс | `Shared/Models/Bouquet.cs` → `AvailableCount` | состояние ресурса видно в ответе `GetBouquetStatus` |
| Тр. 3. Анализ Race Condition | `ReserveWithoutSyncAsync` (режим `None`) | два клиента читают `1`, оба SUCCESS, ресурс `−1` |
| Тр. 4. Выделение критической секции | метод `CheckAndUpdate` | атомарная последовательность «проверка → изменение» |
| Тр. 5. Реализация синхронизации (lock) | `ReserveWithLock` + `private readonly object _syncObject` | `LOCK/UNLOCK` в журнале, секции не пересекаются |
| Тр. 6. Правило границ секции | `CheckAndUpdate` вызывается только внутри `lock`/`await WaitAsync` | проверка не может быть вынесена наружу |
| Тр. 7. Повторный эксперимент | два клиента, ресурс = 1 | **1 SUCCESS + 1 FAIL**, ресурс = 0 |
| Тр. 8. Логирование | `RequestContext` + `Server.Log`, фазы `LOCK/UNLOCK/READ/WRITE/SUCCESS/FAIL` | подробный журнал с временными метками |
| Тр. 9. Другой механизм | проект `Server.Semaphore`, `ReserveWithSemaphoreAsync` | тот же результат 1 SUCCESS + 1 FAIL |
| Тр. 10. Таблица 1 | журнал сервера, раздел 6.1 | заполненные строки по трём механизмам |
| Тр. 11. Unit-тесты | `SynchronizationTests`, `TimeSynchronizationTests` | 33 теста, все проходят |
| Тр. 1. Модель смещения часов | `ClientClock.ClockOffset`, аргумент `--offset` | локальное время отличается от серверного на ±N с |
| Тр. 2. Демонстрация проблемы | пункт меню 9, `GetServerTime` | разница `Local − Server = +7.000 с` |
| Тр. 3. Запрос времени | `GetServerTimeAsync` (операция `GetServerTime`/`TimeRequest`) | TIME_REQUEST → TIME_RESPONSE |
| Тр. 4. Расчёт смещения | `TimeSynchronizer.SyncAsync` | `Offset = −7.001 с` при расхождении 7 с |
| Тр. 5. Измерение RTT | `Rtt = t4 − t1` | RTT = 0.003 с на loopback |
| Тр. 6. Скорректированное время | `_clock.ApplyCorrection(offset)`, `CorrectedNow` | `CorrectedTime ≈ ServerTime`, остаточная ошибка 1 мс |
| Тр. 7. Искусственная задержка | поле `delayMs` в запросе времени | ошибка поправки ≈ `delay/2` (0.5 с при 1 с) |
| Тр. 8. Повторная проверка | `MeasureResidualErrorAsync`, пункт меню 12 | разница после синхронизации |
| Тр. 9. Итоговый эксперимент | пункт меню 14 | три проверки подряд, значения для Таблицы 2 |
| Тр. 11. Обработка ошибок | `ServerUnavailableException`, `TimeSynchronizationException`, `ErrorCodes` | сервер недоступен, разрыв, неизвестная операция, неверное время |

## 12. Ответы на контрольные вопросы (1–8)

1. **Что такое синхронизация потоков?** — Механизмы, обеспечивающие согласованную работу
   нескольких потоков/задач, выполняющихся параллельно: ограничение одновременного доступа
   к общему ресурсу (взаимное исключение), упорядочивание операций и видимость изменений.
2. **Что такое взаимное исключение (mutual exclusion)?** — Требование, согласно которому
   в каждый момент времени критическую секцию выполняет не более одного потока.
3. **Что такое критическая секция?** — Участок программы, в котором выполняются действия
   с общим ресурсом, не допускающие одновременного выполнения несколькими потоками.
4. **Что такое Race Condition?** — Ситуация, когда результат зависит от порядка и момента
   выполнения параллельных операций над общими данными (в нашей ЛР — два успешных
   бронирования одного букета и счётчик, ушедший в минус).
5. **Почему Race Condition возникает при read → check → update?** — Потому что между
   чтением значения и его изменением другой поток может успеть изменить ресурс: решение
   принимается по устаревшему значению, возникает потерянное обновление.
6. **Почему `lock` устраняет Race Condition?** — Пока один поток находится внутри `lock`,
   второй не может войти в тот же блок, поэтому «проверка + изменение» выполняются
   атомарно и решение принимается по актуальному состоянию ресурса.
7. **Что именно помещать внутрь `lock`?** — Только те действия, которые должны быть
   неделимыми: чтение общего состояния, проверку, изменение и подтверждение операции.
8. **Чем `lock` отличается от `Monitor`?** — `lock` — это синтаксический сахар над
   `Monitor.Enter/Exit`: компилятор сам освобождает блокировку в `finally`. `Monitor`
   даёт низкоуровневый доступ (вход/выход вручную, `TryEnter` с таймаутом, `Wait`/`Pulse`).

### 12.1 Ответы на контрольные вопросы (9–16)

9. **Для чего используется `Mutex`?** — Для взаимного исключения **между процессами**
   (именованный объект ядра ОС). Он заметно дороже `lock`, поэтому внутри одного процесса
   его не используют.
10. **Для чего используется `SemaphoreSlim`?** — Для ограничения количества задач,
    одновременно выполняющих критический участок: при значении 1 — взаимное исключение,
    при значении N — «не более N одновременно». В отличие от `lock`, `WaitAsync` не
    блокирует поток, и внутри секции допустим `await`.
11. **Для каких операций подходит `Interlocked`?** — Для простых атомарных операций над
    одной переменной: `Increment`, `Decrement`, `Exchange`, `CompareExchange`, `Add` —
    например, безопасного счётчика запросов. Он **не** заменяет `lock`, если нужно
    атомарно выполнить несколько взаимосвязанных действий.
12. **Почему часы разных компьютеров могут показывать разное время?** — Разные
    источники синхронизации и качество кварцевых генераторов (у каждого узла свой ход),
    сетевые задержки при автоматической настройке, дрейф со временем, а также ручная
    установка и переход на летнее время.
13. **Что такое RTT (Round-Trip Time)?** — Полное время, за которое запрос доходит до
    сервера и ответ возвращается клиенту: `RTT = t4 − t1`. Измеряется по часам клиента,
    поэтому собственное смещение его часов в разности сокращается.
14. **Как оценить задержку передачи сообщения через RTT?** — Приближённо как `RTT / 2`,
    исходя из предположения, что задержки в прямом и обратном направлении примерно
    равны. Отсюда оценка момента фиксации времени сервером: `t1 + RTT/2`.
15. **Почему сетевая задержка снижает точность синхронизации?** — Потому что оценка
    `RTT/2` предполагает симметричность задержек. При асимметрии или при задержке
    обработки на сервере момент фиксации времени смещается относительно середины обмена
    (в нашем эксперименте ошибка ≈ `delay/2`), и поправка получается неточной.
16. **Почему синхронизация времени не гарантирует абсолютно одинаковые часы?** — Из-за
    асимметрии и джиттера сети, дрейфа часов, конечной точности измерений и того, что
    сервер фиксирует время в момент обработки. Синхронизация даёт оценку с погрешностью
    (в наших измерениях — единицы миллисекунд) и требует периодического повторения.

---
## 13. Ключевые выводы

1. **Race Condition возникает не от параллельности как таковой, а от неатомарности.**
   Последовательность «прочитать → проверить → изменить» должна выполняться неделимо;
   защита отдельной строки не помогает.
2. **Критическая секция должна быть минимальной.** Заблокирован только участок работы с
   общим ресурсом — сервер остаётся параллельным: независимые запросы проходят, пока
   критическая секция занята.
3. **`lock` и `SemaphoreSlim` решают одну задачу по-разному.** `lock` проще и дешевле, но
   синхронен (удерживает поток, внутри нельзя `await`); `SemaphoreSlim` асинхронен, не
   блокирует поток и умеет ограничивать параллелизм числом разрешений. Для короткой
   критической секции одного процесса подходят оба.
4. **Журнал — это доказательство.** Метки `LOCK/UNLOCK/READ/WRITE/SUCCESS/FAIL`
   позволяют проверить и параллельность (`START` второго раньше `UNLOCK` первого), и
   взаимное исключение (`LOCK` второго позже `UNLOCK` первого).
5. **В распределённой системе нельзя доверять собственным часам.** Синхронизация строится
   на обмене с эталонным узлом: `RTT = t4 − t1`, `Offset = serverTime − (t1 + RTT/2)`,
   `CorrectedTime = LocalTime + Offset`.
6. **Точность ограничена задержкой сети.** Упрощённая модель с одной меткой сервера
   ошибается примерно на `delay/2` обработки; полноценный NTP использует четыре метки,
   чтобы исключить эту составляющую.
7. **Исправление проверяется экспериментом и тестами:** без синхронизации — два SUCCESS
   при одном ресурсе и счётчик `−1`; с `lock` и `SemaphoreSlim` — ровно один SUCCESS,
   второй получает `OUT_OF_STOCK`, счётчик остаётся `0` (33 unit-теста).